## Full Cleaning Example

In [10]:
import pandas as pd

df = pd.read_csv("../data/amazon_laptop.csv")

df.head()

,Unnamed: 0,product_title,rating,total_rating,price,next_month_bought,offer,mrp,product_details_dict,brand,...,screen_size,colour,hard_disk_size,cpu_model,ram,os,special_features,graphics_card,about,technical_details_dict
0,0,"Lenovo Yoga Slim 7, Intel Core Ultra 5 125H, 1...",4.3,(63),"85,990.",NaN,-28%,"₹81,990","{'Brand': 'Lenovo', 'Model Name': 'Yoga Slim 7...",Lenovo,...,14,Luna Grey,512 GB,Intel Core Ultra 5,16 GB,Windows 11 Home,"Anti Glare Coating, Backlit Keyboard, HD Audio",Integrated,About this itemAI Enabled Processor: Intel Cor...,"{'Brand': '\u200eLenovo', 'Manufacturer': '\u2..."
1,1,Lenovo V15 G4 AMD Ryzen 5 7520U 15.6 inch FHD ...,4.0,(313),"38,000.",500+ boughtin past month,-27%,"₹39,999","{'Brand': 'Lenovo', 'Model Name': 'Lenovo V15'...",Lenovo,...,15.6 Inches,Gray,512 GB,Ryzen 5,16 GB,Windows 11,HD Audio,Integrated,About this itemProcessor: AMD Ryzen 5 7520U pr...,"{'Brand': '\u200eLenovo', 'Manufacturer': '\u2..."
2,2,"BrowseBook 14.1"" FHD IPS Laptop | Best Student...",3.1,(62),"16,990.",200+ boughtin past month,-60%,"₹12,090","{'Brand': 'Neopticon', 'Model Name': 'BrowseBo...",Neopticon,...,14.1 Inches,Gray,128 GB,Celeron N4020,4 GB,Windows 11 Home,"HD Audio, Memory Card Slot",Integrated,About this itemFull HD IPS Display: The 14.1-i...,"{'Brand': '\u200eNeopticon', 'Manufacturer': '..."
3,3,"acer Aspire Lite, AMD Ryzen 3-7330U, 16GB RAM,...",3.9,"(2,071)","24,490.",400+ boughtin past month,-38%,"₹30,990","{'Brand': 'acer', 'Model Name': 'Aspire Lite',...",acer,...,15.6 Inches,Steel Gray,512 GB,Ryzen 3,16 GB,Windows 11 Home,Lightweight,Integrated,About this itemProcessor: AMD Ryzen 3 7330U pr...,"{'Brand': '\u200eacer', 'Manufacturer': '\u200..."
4,4,"ASUS Gaming V16 (2025) 14th Gen,Intel Core 7 2...",4.0,(48),"99,990.",50+ boughtin past month,-17%,"₹94,990","{'Brand': 'ASUS', 'Model Name': 'ASUS Vivobook...",ASUS,...,16 Inches,Black | Core 7 240H,512 GB,Intel Core 7,16 GB,Windows 11 Home,"Anti-glare display, Backlit Keyboard, Precisio...",Integrated,About this itemProcessor : Intel Core 7 Proces...,"{'Brand': '\u200eASUS', 'Manufacturer': '\u200..."


In [11]:
df = df.drop(columns=["Unnamed: 0", "product_title", "next_month_bought",
                     "product_details_dict", "os", "about", "technical_details_dict", 
                      "mrp", "cpu_model", "model_name", "special_features"])

In [12]:
df["rating"] = df["rating"].fillna(df["rating"].median())

In [13]:
df["total_rating"] = (
    df["total_rating"].str.replace(r"([(),])", "", regex=True)
    .astype("float")
    .pipe(lambda x: x.fillna(x.median()))
)

In [14]:
df["price"] = (
    df["price"].str.replace(",", "")
    .astype("float")
    .pipe(lambda x: x.fillna(x.median()))
)

In [15]:
df["offer"] = (
    df["offer"].str.replace("%", "")
    .astype("float")
    .pipe(lambda x: x.fillna(x.median()))
)

In [16]:
CM_TO_IN = 0.393701

screen_size_multiplier = (
    df["screen_size"]
    .str.extract(r"([A-Za-z]+)")
    .map(lambda x: 1 if x == "Inches" else CM_TO_IN if x == "Centimetres" else 1)
)

df["screen_size"] = (
    df["screen_size"].str.replace(r"(\s?[A-Za-z]+)", "", regex=True)
    .astype("float")
    .pipe(lambda x: x.fillna(x.median()))
)*screen_size_multiplier[0]


In [17]:
df["colour"].unique()

array(['Luna Grey', 'Gray', 'Steel Gray', 'Black | Core 7 240H',
       'Black, R7 | RTX 3050', 'Black', 'Mica Silver', 'Silver',
       'Iron Grey', 'R7 7735HS', 'Mecha Gray', 'Natural Silver', 'Blue',
       'Graphite Black', '2024 Aura | Ultra 7 258V', 'Glacier Silver',
       'Platinum Silver', 'Grey', 'Mixed Black', 'Cloud Grey',
       'Off Black', 'Eclipse Gray', 'Ryzen 7-7th Gen-16GB',
       'Glacier silver', 'R7 7735HS / Office 2024', 'Quiet Blue',
       'Performance blue', 'Corei3-13 | Blue', 'Cool Silver',
       'Shadow Black', 'Carbon Grey', 'Core Black', 'Office 2024_Blue',
       '12GB Natural Silver', 'Space Black', 'Green', 'Turbo Silver',
       'Ice Blue', 'Aqua', 'Eclipse Black', 'Blue | R7 5825U',
       'Cloud Grey - Ryzen 5 5500U', '2025 Aura | Ultra 5 226V',
       'PLATINUM_GREY', 'Eclipse Grey', 'Graphite', 'Pure White',
       'Jaeger Gray', 'Arctic Grey', 'Black | RTX 3050 | MSO',
       'Cosmos Gray', 'Scandinavian White', 'Ash Grey', 'Platinum',
       '

In [18]:
import re

df["colour_new"] = df["colour"].str.extract(r"""(Grey|Gray|Black|Silver|Blue|Aura|silver
                                                 |blue|Green|Aqua|GREY|White|Platinum)""", flags=re.VERBOSE)

df["colour_new"] = (
    df["colour_new"].str.replace(r"(Gray|GREY)", "Grey", regex=True)
    .str.replace("blue", "Blue")
    .str.replace("silver", "Silver")
)

df["colour_new"] = df["colour_new"].map({
    "Grey": "Grey",
    "Silver": "Silver",
    "Black": "Black",
    "Blue": "Blue",
    "Platinum": "Silver",
    "Aura": "Other",
    "White": "Other",
    "Green": "Other",
    "Aqua": "Blue"
})

df = df.dropna(subset=["colour_new"])

In [19]:
df.head()

TB_TO_GB = 1024

hard_disk_multiplier = (
    df["hard_disk_size"]
    .str.extract(r"([A-Za-z]+)")
    .map(lambda x: 1 if x == "GB" else TB_TO_GB if x == "TB" else 1)
)

df["hard_disk_size"] = (
    df["hard_disk_size"].str.replace(r"(\s?[A-Za-z]+)", "", regex=True)
    .astype("float")
    .pipe(lambda x: x.fillna(x.median()))
)*hard_disk_multiplier[0]


In [20]:
df["ram"] = df["ram"].str.replace(" GB", "").astype("float").pipe(lambda x: x.fillna(x.median()))

In [21]:
df = df.drop(columns="colour")

In [22]:
df.to_csv("../data/amazon_laptop_cleaned.csv", index=False)